In [2]:
import torch
import string

In [3]:
words = open('names.txt', 'r').read().splitlines()

In [4]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [5]:
cleaned_words = []

for word in words:
    cleaned_words.append('..' + word + '.')

cleaned_words[:10]

['..emma.',
 '..olivia.',
 '..ava.',
 '..isabella.',
 '..sophia.',
 '..charlotte.',
 '..mia.',
 '..amelia.',
 '..harper.',
 '..evelyn.']

In [6]:
for w in cleaned_words[:1]:
    for i in range(2, len(w)):
        print(w[i-2], w[i-1], w[i])

. . e
. e m
e m m
m m a
m a .


In [7]:
stoi = {c:i for i, c in enumerate(string.ascii_lowercase)} | {'.': 26}
itos = {i:c for i, c in enumerate(string.ascii_lowercase)} | {26: '.'}

In [8]:
xs = []
ys = []


for w in cleaned_words:
    for i in range(2, len(w)):
        xs.append(stoi[w[i-2]] * 27 + stoi[w[i-1]])
        ys.append(stoi[w[i]])

ys = torch.tensor(ys)
print(xs[:10])
print(ys[:10])

[728, 706, 120, 336, 324, 728, 716, 389, 305, 237]
tensor([ 4, 12, 12,  0, 26, 14, 11,  8, 21,  8])


In [9]:
import torch.nn.functional as F

xenc = F.one_hot(torch.tensor(xs), num_classes=27*27).float()
xenc[:10]

tensor([[0., 0., 0.,  ..., 0., 0., 1.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [10]:
xenc.shape

torch.Size([228146, 729])

In [11]:
g = torch.Generator().manual_seed(21)
W = torch.randn((729, 27), generator=g, requires_grad=True)

In [ ]:
learning_rate = 100

# training loop
for k in range(500):

    # forward pass
    logits = xenc @ W
    counts = torch.exp(logits)
    prob = counts / counts.sum(1, keepdim=True)
    loss = - prob[torch.arange(len(xs)), ys].log().mean()

    # print loss
    print(loss.item())

    # backprop
    W.grad = None
    loss.backward()

    # update W
    W.data -= learning_rate * W.grad



2.2243216037750244
2.224274158477783
2.224226951599121
2.224179983139038
2.224132776260376
2.224086046218872
2.224039316177368
2.2239928245544434
2.2239463329315186
2.2238998413085938
2.223853826522827
2.2238075733184814
2.223761558532715
2.2237157821655273
2.22367000579834
2.2236242294311523
2.223578929901123
2.2235333919525146
2.2234883308410645
2.223443031311035
2.223397970199585
2.223353147506714
2.2233083248138428
2.2232635021209717
2.2232189178466797
2.223174571990967
2.223130226135254
2.223085880279541
2.2230417728424072
2.2229979038238525
2.2229537963867188
2.222910165786743
2.2228662967681885
2.222822666168213
2.2227792739868164
2.22273588180542
2.2226927280426025
2.222649574279785
2.222606658935547
2.2225635051727295
2.222520589828491
2.222478151321411
2.222435235977173
2.222393035888672
2.222350597381592
2.2223081588745117
2.22226619720459
2.222223997116089
2.222181797027588
2.222140073776245
2.2220983505249023
2.2220566272735596
2.222015142440796
2.2219738960266113
2.221932

In [23]:
g = torch.Generator().manual_seed(2147483647)

# name generator
for _ in range(20):
    word = ''
    i, j = 26, 26
    next_char = ''

    while next_char != 26:
        val = i * 27 + j
        x = F.one_hot(torch.tensor([val]), num_classes=27*27).float()
        p = (x @ W).exp()
        p = p / p.sum(1, keepdim=True)
        next_char = int(torch.multinomial(p, num_samples=1, generator=g).item())
        word += itos[next_char]
        i, j = j, next_char

    print(word[:-1])
    
    

    


dey
na
khilvill
rilayian
marni
obennah
el
lia
sa
maydivikuston
puntand
jario
valey
ae
kalaine
kate
violawhi
zarlettaiddielieus
mata
keesee


In [35]:
# create dev and test sets
indices = torch.randperm(xenc.size(0))
shuffled_data = xenc[indices]
shuffled_y = ys[indices]

train_set, dev_set, test_set = torch.tensor_split(shuffled_data, [int(0.8*xenc.size(0)), int(0.9*xenc.size(0))], dim=0)
train_y, dev_y, test_y = torch.tensor_split(shuffled_y, [int(0.8*xenc.size(0)), int(0.9*xenc.size(0))], dim=0)
print(train_set.shape, dev_set.shape, test_set.shape)

torch.Size([182516, 729]) torch.Size([22815, 729]) torch.Size([22815, 729])


In [51]:
V = torch.randn((729, 27), generator=g, requires_grad=True)

In [58]:
learning_rate = 100
lbd = 0.000001

# training loop
for k in range(500):

    # forward pass
    train_logits = train_set @ V
    train_counts = torch.exp(train_logits)
    train_prob = train_counts / train_counts.sum(1, keepdim=True)
    train_loss = -train_prob[torch.arange(len(train_set)), train_y].log().mean() + lbd*(V**2).sum()

    # print loss
    print(train_loss.item())

    # backprop
    V.grad = None
    train_loss.backward()

    # update W
    V.data -= learning_rate * V.grad

2.2328178882598877
2.23280930519104
2.2328007221221924
2.2327921390533447
2.232783555984497
2.2327747344970703
2.2327659130096436
2.232757329940796
2.2327487468719482
2.2327401638031006
2.232731580734253
2.2327232360839844
2.2327146530151367
2.232706069946289
2.2326972484588623
2.2326886653900146
2.232680320739746
2.2326719760894775
2.23266339302063
2.2326550483703613
2.2326464653015137
2.232638120651245
2.2326295375823975
2.23262095451355
2.2326128482818604
2.2326042652130127
2.232595920562744
2.2325875759124756
2.232578992843628
2.2325706481933594
2.2325620651245117
2.2325539588928223
2.232545852661133
2.232537269592285
2.2325289249420166
2.232520818710327
2.2325124740600586
2.232503890991211
2.2324957847595215
2.232487678527832
2.2324793338775635
2.232471227645874
2.2324628829956055
2.232454538345337
2.2324466705322266
2.232438325881958
2.2324302196502686
2.232421875
2.2324135303497314
2.232405662536621
2.2323973178863525
2.232389450073242
2.2323813438415527
2.232372999191284
2.2323

In [60]:
# calculating loss on dev set and train sets
dev_logits, test_logits = dev_set @ V, test_set @ V
dev_counts, test_counts = torch.exp(dev_logits), torch.exp(test_logits)
dev_prob, test_prob = dev_counts / dev_counts.sum(1, keepdim=True), test_counts / test_counts.sum(1, keepdim=True)
dev_loss, test_loss = -dev_prob[torch.arange(len(dev_set)), dev_y].log().mean(), -test_prob[torch.arange(len(test_set)), test_y].log().mean()

# print loss
print("dev loss:", dev_loss.item())
print("test loss:", test_loss.item())

dev loss: 2.239628553390503
test loss: 2.221823215484619
